# 配套实践 12-01：条件 Diffusion 从噪声恢复双峰动作

本练习在二维动作空间训练一个小型条件 Diffusion。Context 0 的动作位于左侧两个模式，Context 1 位于右侧两个模式。我们先观察前向加噪怎样破坏结构，再训练噪声预测网络并逐步生成条件动作。二维点代表展平 Action Chunk 的最小类比，算法接口与高维动作相同。依赖：PyTorch、NumPy、Matplotlib；CPU 即可运行。

<a href="https://qi-robotics.github.io/robot-world-model-tutorial/intermediate/12-diffusion-and-flow-matching/" target="_blank">在新标签页返回课程正文</a>

In [ ]:
import math  # 计算生成时间的正弦与余弦 embedding
import numpy as np  # 整理绘图网格和损失曲线
import torch  # 构造加噪过程并训练噪声预测网络
from torch import nn  # 使用多层感知机和激活函数
import matplotlib.pyplot as plt  # 绘制数据分布、加噪过程和反向样本
torch.set_num_threads(2)  # 限制轻量实验的 CPU 线程开销
torch.manual_seed(12)  # 固定训练数据、网络初始化和反向采样
np.random.seed(12)  # 固定 NumPy 侧随机过程
plt.rcParams["figure.dpi"] = 120  # 提高笔记本图像显示清晰度

## 1. 数据分布与前向加噪

每个 Context 都有上下两个合理动作模式，目标条件只决定模式位于左侧还是右侧。Diffusion 使用 50 个离散噪声步；下面对同一批干净动作直接计算不同步的带噪结果。

In [ ]:
def sample_action_data(sample_count):  # 定义从条件双峰动作分布采样的函数
    contexts = torch.randint(0, 2, (sample_count,))  # 为每个样本随机选择左侧或右侧 Context
    mode_signs = torch.where(torch.rand(sample_count) > 0.5, 1.0, -1.0)  # 为每个 Context 随机选择上方或下方模式
    horizontal_centers = torch.where(contexts == 0, -1.5, 1.5)  # 根据 Context 设置动作横坐标中心
    centers = torch.stack([horizontal_centers, mode_signs], dim=1)  # 组合四个可能的条件模式中心
    actions = centers + 0.12 * torch.randn(sample_count, 2)  # 在模式中心附近加入示范变化
    return actions, contexts  # 返回二维动作和对应 Context
diffusion_steps = 50  # 设置离散 Diffusion 的总噪声步数
betas = torch.linspace(0.0005, 0.15, diffusion_steps)  # 使用由小到大的线性噪声日程
alphas = 1.0 - betas  # 计算每一步保留信号的比例
alpha_bars = torch.cumprod(alphas, dim=0)  # 计算从干净动作累计到各步的信号比例
clean_actions, clean_contexts = sample_action_data(1200)  # 采样一批用于展示的干净条件动作
fixed_noise = torch.randn_like(clean_actions)  # 为所有噪声等级固定同一组 Gaussian 噪声
shown_steps = [0, 9, 24, 49]  # 选择四个噪声等级展示结构消失过程
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2), sharex=True, sharey=True)  # 创建四幅前向加噪分布图
for axis, step_index in zip(axes, shown_steps):  # 依次计算并绘制四个噪声等级
    noisy_actions = alpha_bars[step_index].sqrt() * clean_actions + (1.0 - alpha_bars[step_index]).sqrt() * fixed_noise  # 使用闭式公式得到当前带噪动作
    for context_value, color in zip([0, 1], ["#2563eb", "#ea580c"]):  # 分别绘制两个任务条件
        selected = clean_contexts == context_value  # 找出当前 Context 的样本
        axis.scatter(noisy_actions[selected, 0], noisy_actions[selected, 1], s=7, alpha=0.35, color=color)  # 显示当前条件下的带噪动作
    axis.set(title=f"Noise step {step_index}", xlabel="Action dim 1", xlim=(-3.2, 3.2), ylim=(-2.8, 2.8))  # 标注噪声步和动作坐标范围
axes[0].set_ylabel("Action dim 2")  # 为共用纵轴标记第二动作维度
fig.suptitle("Forward diffusion gradually removes conditional action structure")  # 强调前向过程逐步破坏数据结构
fig.tight_layout()  # 调整四幅子图间距
plt.show()  # 显示动作分布的前向加噪过程

**怎样理解结果：** 第 0 步仍能看见左右条件与上下模式；噪声等级增大后，四簇逐渐重叠，到第 49 步已接近无结构 Gaussian。每个噪声步都可由干净动作一次计算得到，训练不需要依次运行前 49 步。

## 2. 学习在任意噪声等级预测噪声

网络输入包括二维带噪动作、三个生成时间特征和两维 Context one-hot，共 7 维。每次更新重新采样干净动作、噪声步和 Gaussian 噪声，目标是恢复这次实际加入的噪声。

In [ ]:
class NoisePredictor(nn.Module):  # 定义条件噪声预测网络
    def __init__(self):  # 初始化三层小型感知机
        super().__init__()  # 初始化 PyTorch 模型基类
        self.network = nn.Sequential(nn.Linear(7, 64), nn.SiLU(), nn.Linear(64, 64), nn.SiLU(), nn.Linear(64, 2))  # 把动作、时间和 Context 映射回二维噪声
    def forward(self, noisy_actions, step_indices, contexts):  # 定义任意噪声等级的条件前向计算
        normalized_steps = step_indices.float() / (diffusion_steps - 1)  # 把离散噪声步缩放到零到一
        time_features = torch.stack([normalized_steps, torch.sin(2.0 * math.pi * normalized_steps), torch.cos(2.0 * math.pi * normalized_steps)], dim=1)  # 建立线性与周期生成时间特征
        context_features = nn.functional.one_hot(contexts, num_classes=2).float()  # 把两个任务条件变成 one-hot 向量
        model_inputs = torch.cat([noisy_actions, time_features, context_features], dim=1)  # 拼接带噪动作、噪声等级和 Context
        return self.network(model_inputs)  # 输出与动作形状相同的噪声预测
noise_predictor = NoisePredictor()  # 创建待训练的条件 Diffusion 网络
optimizer = torch.optim.Adam(noise_predictor.parameters(), lr=0.001)  # 使用 Adam 更新噪声预测参数
loss_history = []  # 保存每次更新的噪声预测 MSE
for update_index in range(3000):  # 使用三千个轻量批次训练网络
    clean_batch, context_batch = sample_action_data(256)  # 采样一批干净条件动作
    step_batch = torch.randint(0, diffusion_steps, (256,))  # 为每个动作独立选择噪声等级
    noise_batch = torch.randn_like(clean_batch)  # 采样本次真正加入的 Gaussian 噪声
    noisy_batch = alpha_bars[step_batch, None].sqrt() * clean_batch + (1.0 - alpha_bars[step_batch, None]).sqrt() * noise_batch  # 一次构造任意步的带噪动作
    predicted_noise = noise_predictor(noisy_batch, step_batch, context_batch)  # 根据带噪动作、时间和条件预测噪声
    loss = ((predicted_noise - noise_batch) ** 2).mean()  # 计算预测噪声与真实噪声的均方误差
    optimizer.zero_grad()  # 清除上一个批次残留的梯度
    loss.backward()  # 反向传播计算噪声网络的参数梯度
    optimizer.step()  # 更新网络参数以改善所有噪声等级的预测
    loss_history.append(float(loss.detach()))  # 保存当前更新的训练损失
smoothed_loss = np.convolve(loss_history, np.ones(80) / 80.0, mode="valid")  # 使用滑动平均显示总体收敛趋势
fig, axis = plt.subplots(figsize=(8.8, 3.6))  # 创建噪声预测损失曲线
axis.plot(np.arange(len(smoothed_loss)) + 79, smoothed_loss, color="#2563eb")  # 绘制平滑后的训练 MSE
axis.set(title="Noise prediction improves across random diffusion steps", xlabel="Update", ylabel="Noise MSE")  # 标注优化目标和横轴含义
axis.grid(alpha=0.2)  # 添加淡网格帮助观察收敛趋势
fig.tight_layout()  # 调整图像边距
plt.show()  # 显示条件 Diffusion 的训练过程

**怎样理解结果：** 平滑损失从接近 1 明显下降，说明网络能利用动作结构、噪声等级和 Context 估计加入的噪声。损失不会降到零，因为高噪声位置仅凭一个样本不能唯一恢复原动作；模型学习的是数据分布提供的统计去噪方向。

## 3. 从同一噪声分布反向生成条件动作

从标准 Gaussian 初始化左右两组粒子，按 49 到 0 的顺序应用 DDPM 反向均值与后验噪声。我们保存几个中间状态，观察 Context 如何让最终样本落到不同横向位置，同时保留上下两个模式。

In [ ]:
@torch.no_grad()  # 关闭采样过程的梯度记录以降低开销
def sample_with_snapshots(sample_count, context_value):  # 定义保存反向去噪中间状态的条件采样函数
    current_actions = torch.randn(sample_count, 2)  # 从标准 Gaussian 初始化动作粒子
    context_batch = torch.full((sample_count,), context_value, dtype=torch.long)  # 为全部粒子设置同一任务 Context
    snapshot_indices = {49, 34, 19, 4, 0}  # 选择五个反向阶段记录粒子位置
    snapshots = {}  # 准备按噪声步保存粒子副本
    for step_index in range(diffusion_steps - 1, -1, -1):  # 从最大噪声步逐次运行到干净动作
        step_batch = torch.full((sample_count,), step_index, dtype=torch.long)  # 为当前批次建立统一噪声步张量
        predicted_noise = noise_predictor(current_actions, step_batch, context_batch)  # 估计当前动作中的噪声分量
        reverse_mean = (current_actions - betas[step_index] / torch.sqrt(1.0 - alpha_bars[step_index]) * predicted_noise) / torch.sqrt(alphas[step_index])  # 计算 DDPM 反向分布均值
        if step_index > 0:  # 判断当前是否还需要保留反向随机性
            posterior_variance = betas[step_index] * (1.0 - alpha_bars[step_index - 1]) / (1.0 - alpha_bars[step_index])  # 计算当前步的后验方差
            current_actions = reverse_mean + posterior_variance.sqrt() * torch.randn_like(current_actions)  # 从当前反向分布采样下一步动作
        else:  # 处理最后一步不再添加随机噪声的情况
            current_actions = reverse_mean  # 使用最终反向均值得到干净动作
        if step_index in snapshot_indices:  # 检查当前阶段是否需要用于可视化
            snapshots[step_index] = current_actions.clone()  # 保存不会被后续更新修改的动作副本
    return snapshots  # 返回五个反向阶段的条件动作分布
left_snapshots = sample_with_snapshots(700, 0)  # 为 Context 0 生成左侧双峰动作
right_snapshots = sample_with_snapshots(700, 1)  # 为 Context 1 生成右侧双峰动作
display_steps = [49, 34, 19, 4, 0]  # 按反向生成顺序排列五个快照
fig, axes = plt.subplots(1, 5, figsize=(14, 3.1), sharex=True, sharey=True)  # 创建五幅反向生成阶段图
for axis, step_index in zip(axes, display_steps):  # 依次绘制每个反向噪声阶段
    left_points = left_snapshots[step_index]  # 读取左侧 Context 的当前粒子
    right_points = right_snapshots[step_index]  # 读取右侧 Context 的当前粒子
    axis.scatter(left_points[:, 0], left_points[:, 1], s=7, alpha=0.28, color="#2563eb", label="Context 0")  # 绘制蓝色左条件样本
    axis.scatter(right_points[:, 0], right_points[:, 1], s=7, alpha=0.28, color="#ea580c", label="Context 1")  # 绘制橙色右条件样本
    axis.set(title=f"Reverse step {step_index}", xlabel="Action dim 1", xlim=(-3.2, 3.2), ylim=(-2.8, 2.8))  # 标注当前反向步和动作范围
axes[0].set_ylabel("Action dim 2")  # 为共用纵轴标记第二动作维度
axes[-1].legend(loc="upper right", fontsize=8)  # 在最终结果图显示两个 Context 的颜色
fig.suptitle("Conditioned reverse diffusion turns noise into four action modes")  # 强调条件去噪恢复多峰动作
fig.tight_layout()  # 调整五幅子图间距
plt.show()  # 显示完整反向 Diffusion 生成过程

**怎样理解结果：** 初始粒子对两个 Context 都来自同一 Gaussian；随着反向步骤推进，蓝色粒子整体移向左侧、橙色粒子移向右侧，并分别裂成上下两个模式。最终样本不是训练点的逐个复制，而是分布附近的新动作。

**本练习的结论：** 条件 Diffusion 用同一网络在多个噪声等级提供局部去噪方向，经过许多次调用才得到动作。扩展到 Action Chunk 时，只需把二维点换成时间×动作维度张量，但网络结构、mask、物理边界和采样延迟会变得更加重要。